# 04 · Feature Engineering

Urutannya: split dulu, baru semua yang belajar dari data (median pengisi, daftar district, kolom one-hot) dihitung dari train saja. Kalau dibalik, informasi data test bocor ke train.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

In [2]:
df = pd.read_csv("../data/processed/rumah_clean.csv")
TARGET = "price_in_rp"
print(df.shape)

(3172, 24)


In [3]:
BUANG = ["url", "ads_id", "title", "address", "property_type", "facilities"]
df = df.drop(columns=BUANG)

In [4]:
df["listrik"] = pd.to_numeric(df["electricity"].str.extract(r"(\d+)")[0], errors="coerce")
df = df.drop(columns="electricity")

In [5]:
df["certificate"] = df["certificate"].replace(
    {"hp - hak pakai": "lainnya (ppjb,girik,adat,dll)"}
)
print(df["certificate"].value_counts(dropna=False))

certificate
shm - sertifikat hak milik       2672
lainnya (ppjb,girik,adat,dll)     189
hgb - hak guna bangunan           181
NaN                               130
Name: count, dtype: int64


In [6]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# rumah "mirip kembar" (kecamatan, harga, luas, kamar sama) diberi nomor grup yang sama,
# supaya satu grup tidak terpecah ke train DAN test (mencegah model "menghafal" jawaban)
KUNCI_GRUP = ["district", TARGET, "land_size_m2", "building_size_m2", "bedrooms", "bathrooms"]
grup = df.groupby(KUNCI_GRUP, dropna=False).ngroup()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(X, y, groups=grup))

X_train, X_test = X.iloc[idx_train].copy(), X.iloc[idx_test].copy()
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
grup_train = grup.iloc[idx_train]          # dipakai lagi untuk GroupKFold di tahap modeling

print(X_train.shape, X_test.shape)
print("grup yang muncul di train DAN test:", len(set(grup.iloc[idx_train]) & set(grup.iloc[idx_test])))

(2533, 17) (639, 17)
grup yang muncul di train DAN test: 0


Split pakai `GroupShuffleSplit`: rumah yang mirip kembar (kecamatan, harga, luas, dan kamar sama) dijadikan satu grup, dan satu grup selalu masuk ke sisi yang sama. Hasilnya 2.533 train / 639 test dengan 0 grup yang terpecah.

In [7]:
(X_train.isnull().mean() * 100).round(2).sort_values(ascending=False).head(10)

building_orientation    44.89
building_age            37.58
furnishing              11.05
listrik                  8.33
property_condition       6.55
certificate              4.54
bedrooms                 1.03
bathrooms                0.87
floors                   0.12
district                 0.00
dtype: float64

In [8]:
KAT_KOSONG = ["certificate", "furnishing", "property_condition", "building_orientation"]
for col in KAT_KOSONG:
    X_train[col] = X_train[col].fillna("tidak diketahui")
    X_test[col] = X_test[col].fillna("tidak diketahui")

In [9]:
NUM_PENANDA = ["building_age", "listrik"]
NUM_SEDIKIT = ["bedrooms", "bathrooms", "floors"]

# 1) hitung median dari TRAIN saja
nilai_isi = {col: X_train[col].median() for col in NUM_PENANDA + NUM_SEDIKIT}

# 2) buat penanda "dulunya kosong" SEBELUM diisi
for col in NUM_PENANDA:
    X_train[col + "_kosong"] = X_train[col].isnull().astype(int)
    X_test[col + "_kosong"] = X_test[col].isnull().astype(int)

# 3) isi, pakai angka yang sama untuk train dan test
for col, v in nilai_isi.items():
    X_train[col] = X_train[col].fillna(v)
    X_test[col] = X_test[col].fillna(v)

print(nilai_isi)

{'building_age': np.float64(1.0), 'listrik': np.float64(2200.0), 'bedrooms': np.float64(3.0), 'bathrooms': np.float64(2.0), 'floors': np.float64(2.0)}


Sebelum diisi median, dibuat kolom penanda `_kosong` untuk building_age dan listrik, supaya model tetap tahu mana nilai yang aslinya tidak diketahui.

In [10]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

0 0


In [11]:
hitung = X_train["district"].value_counts()
district_ramai = hitung[hitung >= 10].index      # dihitung dari TRAIN

for d in (X_train, X_test):
    d["district"] = d["district"].where(d["district"].isin(district_ramai), "lainnya")

print(len(district_ramai), "district dipertahankan")
print("porsi 'lainnya' di train:", (X_train["district"] == "lainnya").mean().round(3))

65 district dipertahankan
porsi 'lainnya' di train: 0.313


District yang muncul kurang dari 10 kali di train digabung menjadi "lainnya" (65 district dipertahankan). Terlalu sedikit data untuk dipelajari sendiri-sendiri.

In [12]:
KAT = ["city", "district", "certificate", "furnishing",
       "property_condition", "building_orientation"]

X_train_enc = pd.get_dummies(X_train, columns=KAT).astype(float)
X_test_enc = pd.get_dummies(X_test, columns=KAT).astype(float)

# samakan kolom test dengan train
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
print(X_train_enc.shape, X_test_enc.shape)

(2533, 111) (639, 111)


In [13]:
# LightGBM menolak spasi, koma, dan kurung di nama kolom -> ganti dengan "_"
X_train_enc.columns = X_train_enc.columns.str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
X_test_enc.columns = X_test_enc.columns.str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)

print("nama kolom kembar:", X_train_enc.columns.duplicated().sum())
print([c for c in X_train_enc.columns if c.startswith("certificate")])

nama kolom kembar: 0
['certificate_hgb_hak_guna_bangunan', 'certificate_lainnya_ppjb_girik_adat_dll_', 'certificate_shm_sertifikat_hak_milik', 'certificate_tidak_diketahui']


LightGBM menolak spasi, koma, dan kurung di nama kolom, jadi semuanya diganti `_`. Hasil akhir: 111 kolom fitur.

In [14]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
# kembali ke rupiah nanti: np.expm1(prediksi)

In [15]:
import os, joblib
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

X_train_enc.to_csv("../data/processed/X_train.csv", index=False)
X_test_enc.to_csv("../data/processed/X_test.csv", index=False)
y_train_log.to_csv("../data/processed/y_train_log.csv", index=False)
y_test_log.to_csv("../data/processed/y_test_log.csv", index=False)
grup_train.to_csv("../data/processed/grup_train.csv", index=False)

# semua aturan preprocessing, dipakai lagi nanti oleh aplikasi Streamlit
joblib.dump({
    "kolom_buang": BUANG,
    "regex_listrik": r"(\d+)",
    "map_sertifikat": {"hp - hak pakai": "lainnya (ppjb,girik,adat,dll)"},
    "kat_kosong": KAT_KOSONG,
    "num_penanda": NUM_PENANDA,
    "nilai_isi": nilai_isi,
    "district_ramai": list(district_ramai),
    "kolom_kategori": KAT,
    "kolom": list(X_train_enc.columns),
}, "../models/preprocess_info.joblib")

['../models/preprocess_info.joblib']

`preprocess_info.joblib` menyimpan semua aturan di atas, supaya `src/predict.py` bisa memproses input baru dengan cara yang persis sama.